# HVAC Takeoff â€” v10 Training on Kaggle (153 projects)

**Setup:**
1. Upload `yolo_dataset.zip` (1.27 GB) as a Kaggle Dataset first
   - Go to kaggle.com â†’ Datasets â†’ New Dataset â†’ upload the zip
   - Name it `hvac-yolo-dataset-v10`
2. In this notebook: Settings â†’ Accelerator â†’ **GPU P100**
3. Add the dataset: right panel â†’ Add Data â†’ search `hvac-yolo-dataset-v10`
4. Run all cells

In [ ]:
# Step 1: Install
!pip install ultralytics -q

In [ ]:
# Step 2: Copy dataset from Kaggle input to writable directory
import yaml, os, shutil, glob

# Find the dataset â€” Kaggle auto-extracts it and the path varies by user
candidates = glob.glob('/kaggle/input/**/yolo_dataset', recursive=True)
if not candidates:
    print('ERROR: yolo_dataset folder not found in /kaggle/input/')
    print('Listing /kaggle/input/ contents:')
    for root, dirs, files in os.walk('/kaggle/input/'):
        print(f'  {root}/')
        if len(root.replace('/kaggle/input/', '').split('/')) > 4:
            continue
    raise FileNotFoundError('Dataset not found')

src = candidates[0]
dst = '/kaggle/working/yolo_dataset'
print(f'Found dataset at: {src}')

if os.path.exists(dst):
    shutil.rmtree(dst)
print('Copying to writable directory (this takes ~2-3 min)...')
shutil.copytree(src, dst)

# Fix paths in dataset.yaml
config_path = f'{dst}/dataset.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)
config['path'] = dst
with open(config_path, 'w') as f:
    yaml.dump(config, f)

# Clear stale cache files (they contain old paths)
for cache in ['train.cache', 'val.cache']:
    p = f'{dst}/labels/{cache}'
    if os.path.exists(p):
        os.remove(p)

train = len(os.listdir(f'{dst}/images/train'))
val = len(os.listdir(f'{dst}/images/val'))
print(f'\nDataset ready: {train} train, {val} val')
print(f'Classes: {len(config["names"])}')
for i, n in config['names'].items():
    print(f'  {i}: {n}')

In [ ]:
# Step 3: Train YOLOv8s with checkpoint callback
from ultralytics import YOLO
import shutil, os

model = YOLO('yolov8s.pt')

# Callback: copy best.pt to /kaggle/working/ after every epoch
# This way even if training gets killed, the best model so far is preserved
def save_checkpoint(trainer):
    best = '/kaggle/working/runs/hvac_v10/weights/best.pt'
    if os.path.exists(best):
        shutil.copy(best, '/kaggle/working/hvac_yolov8s_v10.pt')
        print(f'  [checkpoint saved: epoch {trainer.epoch + 1}]')

model.add_callback('on_fit_epoch_end', save_checkpoint)

results = model.train(
    data='/kaggle/working/yolo_dataset/dataset.yaml',
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    device=0,
    workers=2,
    project='/kaggle/working/runs',
    name='hvac_v10',
    exist_ok=True,
    flipud=0.0,
    mosaic=0.5,
    scale=0.3,
    save_period=1,     # Save checkpoint every epoch (backup)
)

print('\nTraining complete!')
print(f'Final model at: /kaggle/working/hvac_yolov8s_v10.pt')

In [ ]:
# Step 4: Check results
import pandas as pd

df = pd.read_csv('/kaggle/working/runs/hvac_v10/results.csv')
df.columns = [c.strip() for c in df.columns]
best = df.loc[df['metrics/mAP50(B)'].idxmax()]
print(f"Best epoch: {int(best['epoch'])}")
print(f"Precision: {best['metrics/precision(B)']:.1%}")
print(f"Recall:    {best['metrics/recall(B)']:.1%}")
print(f"mAP50:     {best['metrics/mAP50(B)']:.1%}")
print(f"mAP50-95:  {best['metrics/mAP50-95(B)']:.1%}")

# Show last 10 epochs
print('\nLast 10 epochs:')
print(df[['epoch','metrics/precision(B)','metrics/recall(B)','metrics/mAP50(B)']].tail(10).to_string(index=False))

In [ ]:
# Step 5: Save model
import shutil

best_pt = '/kaggle/working/runs/hvac_v10/weights/best.pt'
out_path = '/kaggle/working/hvac_yolov8s_v10.pt'
shutil.copy(best_pt, out_path)
print(f'Model saved: {out_path}')
print(f'Size: {os.path.getsize(out_path)/1024/1024:.1f} MB')
print()
print('Download from the Output tab on the right panel â†’')
print('Then put it in hvac-takeoff-tool/models/')